# Telco Customer Churn — Prédiction du churn

Pipeline ML de bout en bout : nettoyage → EDA → preprocessing anti-fuite → modélisation
(régression logistique vs XGBoost) → évaluation orientée *recall* → interprétabilité SHAP →
déploiement FastAPI. Décisions et justifications : voir `cahier des charges.md`.

## 1. Chargement des données

In [1]:
import pandas as pd

# Définition des données d'analyse
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(df.shape)
print(df.head())
df.info()
print(df["Churn"].value_counts(normalize=True))

# Identification des lignes problématiques avant nettoyage
mask_bad = pd.to_numeric(df["TotalCharges"], errors="coerce").isna()
print(f"Lignes avec TotalCharges non convertible : {mask_bad.sum()}")
df.loc[mask_bad, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


## 2. Nettoyage

`TotalCharges` est typée `str` : 11 cellules vides, **toutes des clients `tenure = 0`** (jamais facturés). On convertit en numérique puis on impute **0** sur ces lignes — la valeur manquante a une **cause logique identifiée** (zéro mois ⇒ zéro facturé), pas une approximation arbitraire.

Ces deux opérations sont **déterministes** (aucun paramètre appris sur la distribution), donc applicables avant le split sans fuite ; l'imputation *statistique* reste, elle, dans les pipelines (`fit` par fold, §4). `customerID` est écarté des features en §4. Contrôle des doublons exacts au passage.

In [ ]:
# Section 2 — Nettoyage
# TotalCharges : str -> numérique. Les 11 cellules vides deviennent NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Garde-fou : tout NaN généré doit être un client tenure=0 (cause logique identifiée).
# Si un NaN apparaissait ailleurs, l'imputation à 0 ne serait plus justifiée.
nan_idx = df["TotalCharges"].isna()
assert (df.loc[nan_idx, "tenure"] == 0).all(), "NaN TotalCharges hors tenure=0 : a investiguer"

# Règle métier : 0 mois d'ancienneté => 0 facturé. Imputation déterministe (pas de fuite).
df.loc[df["tenure"] == 0, "TotalCharges"] = 0.0

# Doublons exacts (ligne entière, customerID compris) : anomalie de collecte éventuelle.
n_dupes = int(df.duplicated().sum())

print("TotalCharges dtype       :", df["TotalCharges"].dtype)
print("NaN TotalCharges restants:", int(df["TotalCharges"].isna().sum()))
print("Clients tenure=0 -> 0    :", int((df["tenure"] == 0).sum()))
print("Doublons exacts          :", n_dupes)
print("Shape (inchangee)        :", df.shape)

## 3. Analyse exploratoire (EDA)

Analyse **descriptive** sur le jeu nettoyé complet (`df`) : aucun paramètre appris, aucune feature créée ou supprimée → pas de fuite, et le split (§4) reste la frontière des transformations apprenantes. L'analyse des **interactions** est reportée à l'interprétabilité **SHAP** (plus loin), comme prévu au cahier.

- **3.1 Catégorielles** — taux de churn par modalité + test du **Chi²**. Avec n = 7043 presque tout ressort « significatif » : on classe donc par **V de Cramér** (taille d'effet), pas par p-value.
- **3.2 Numériques** (`tenure`, `MonthlyCharges`, `TotalCharges`) — distributions par classe, corrélations, détection formalisée des outliers (règle IQR de Tukey).
- **3.3 Synthèse** — insights business pour le README.

In [ ]:
# 3.1 — Catégorielles : association avec le churn (Chi² + V de Cramér), puis visualisation
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# Catégorielles = non-numériques hors id/cible. exclude="number" écarte SeniorCitizen (0/1,
# numérique au cahier) et évite l'ambiguïté object/str de pandas 3.
cat_cols = df.drop(columns=["customerID", "Churn"]).select_dtypes(exclude="number").columns.tolist()
churn = (df["Churn"] == "Yes").astype(int)
global_rate = churn.mean()

def cramers_v(confusion):
    """V de Cramér corrigé du biais (Bergsma 2013) : taille d'effet dans [0, 1]."""
    chi2 = chi2_contingency(confusion)[0]
    n = confusion.to_numpy().sum()
    r, k = confusion.shape
    phi2corr = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2corr / min(kcorr - 1, rcorr - 1))

rows = []
for col in cat_cols:
    ct = pd.crosstab(df[col], df["Churn"])
    chi2, p, _, _ = chi2_contingency(ct)
    rows.append({"feature": col, "modalites": df[col].nunique(),
                 "cramers_v": round(cramers_v(ct), 3), "chi2": round(chi2, 1), "p_value": p})

chi2_table = pd.DataFrame(rows).sort_values("cramers_v", ascending=False).reset_index(drop=True)
print(chi2_table.assign(p_value=chi2_table["p_value"].map("{:.1e}".format)).to_string(index=False))

# Taux de churn par modalité — panneaux ordonnés par V de Cramér décroissant
ordered = chi2_table["feature"].tolist()
nrows = -(-len(ordered) // 3)
fig, axes = plt.subplots(nrows, 3, figsize=(16, 3.4 * nrows))
axes = axes.ravel()
for ax, col in zip(axes, ordered):
    rate = churn.groupby(df[col]).mean().sort_values()
    ax.bar(range(len(rate)), rate.values, color="#c0392b")
    ax.axhline(global_rate, color="grey", ls="--", lw=1)
    ax.set_xticks(range(len(rate)), rate.index.astype(str), rotation=25, ha="right", fontsize=7)
    v = chi2_table.loc[chi2_table["feature"] == col, "cramers_v"].iat[0]
    ax.set_title(f"{col}  (V={v:.2f})", fontsize=10)
    ax.set_ylim(0, 1)
    for i, val in enumerate(rate.values):
        ax.text(i, val + 0.02, f"{val:.0%}", ha="center", fontsize=7)
for ax in axes[len(ordered):]:
    ax.set_visible(False)
fig.suptitle(f"Taux de churn par modalité — ligne grise = taux global {global_rate:.1%}", y=1.0, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# 3.2 — Numériques : distributions par classe (No churn vs Churn)
num_cont = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(len(num_cont), 2, figsize=(14, 3.8 * len(num_cont)))
for i, col in enumerate(num_cont):
    axes[i, 0].hist(df.loc[churn == 0, col], bins=40, density=True, alpha=0.6,
                    color="#2980b9", label="No churn")
    axes[i, 0].hist(df.loc[churn == 1, col], bins=40, density=True, alpha=0.6,
                    color="#c0392b", label="Churn")
    axes[i, 0].set_title(f"{col} — densité par classe")
    axes[i, 0].legend()

    axes[i, 1].boxplot([df.loc[churn == 0, col], df.loc[churn == 1, col]])
    axes[i, 1].set_xticks([1, 2], ["No churn", "Churn"])
    axes[i, 1].set_title(f"{col} — boxplot par classe")

fig.tight_layout()
plt.show()

In [ ]:
# 3.2 — Corrélations (numériques + cible) et détection formalisée des outliers (IQR)
corr_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
corr = df[corr_cols].assign(Churn=churn).corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Corrélation de Pearson")
fig.tight_layout()
plt.show()

# Outliers : règle IQR de Tukey (k = 1.5). On les détecte et quantifie, on ne les supprime pas
# (valeurs métier plausibles ; XGBoost y est robuste, la LR passe par un StandardScaler).
num_cont = ["tenure", "MonthlyCharges", "TotalCharges"]
out_rows = []
for col in num_cont:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < low) | (df[col] > high)).sum())
    out_rows.append({"feature": col, "Q1": round(q1, 1), "Q3": round(q3, 1),
                     "borne_basse": round(low, 1), "borne_haute": round(high, 1),
                     "n_outliers": n_out, "pct": f"{n_out / len(df):.2%}"})
print(pd.DataFrame(out_rows).to_string(index=False))

### 3.3 Synthèse — insights business

1. **L'engagement contractuel domine.** `Contract` est le 1er facteur (V de Cramér = 0.41) : le *month-to-month* churne très au-dessus de la moyenne, les contrats 1–2 ans très en dessous. Levier de rétention n°1.
2. **Le bloc sécurité/support pèse.** L'absence d'`OnlineSecurity`, `TechSupport`, `OnlineBackup`, `DeviceProtection` (V ≈ 0.28–0.35) est associée à un sur-churn. Piste produit : bundles de fidélisation.
3. **Profil de facturation à risque.** `Fiber optic`, `Electronic check` et `Paperless billing` concentrent le churn ; côté numérique, **faible `tenure`** et **`MonthlyCharges` élevées** sont les marqueurs les plus nets (cf. 3.2). Cible prioritaire : nouveaux clients à facture élevée.
4. **Variables non discriminantes.** `gender` (V ≈ 0, p = 0.49) et `PhoneService` (p = 0.34) : aucune association significative avec le churn — à ne pas sur-interpréter, et utile à mentionner (anti-biais « tout compte »).

> ⚠️ `TotalCharges` est mécaniquement corrélée à `tenure` (cf. matrice 3.2) : redondance attendue, gérée nativement par les arbres ; à garder en tête pour la lecture des coefficients de la régression logistique.

## 4. Feature engineering & preprocessing

La frontière anti-fuite structure tout : on **split d'abord**, puis toute transformation
qui apprend un paramètre (`fit`) ne voit que le train.

Deux préprocesseurs distincts, car la bonne transformation dépend du modèle aval :
- **Régression logistique** : one-hot (pas de faux ordre) + StandardScaler (sensible aux magnitudes).
- **XGBoost** : encodage ordinal entier, aucun scaling (arbres invariants aux transfos monotones).

In [ ]:
# Cible : Churn Yes/No -> 1/0
y = (df["Churn"] == "Yes").astype(int)

# Features : on exclut l'identifiant (non prédictif) et la cible
X = df.drop(columns=["customerID", "Churn"])

# Séparation automatique numérique / catégoriel (zéro liste hardcodée)
num_features = X.select_dtypes(include="number").columns.tolist()
cat_features = X.select_dtypes(include="object").columns.tolist()

print("Numériques  :", num_features)
print("Catégorielles :", cat_features)
print(f"\n{len(num_features)} num + {len(cat_features)} cat = {X.shape[1]} features")
print("Taux de churn global :", round(y.mean(), 4))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # proportions de churn préservées des deux côtés
    random_state=42,
)

print("Train :", X_train.shape, "| churn =", round(y_train.mean(), 4))
print("Test  :", X_test.shape,  "| churn =", round(y_test.mean(), 4))

In [ ]:
#Pretraitement des données pour le modèle qui va utiliser la regression Logistique

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipe_lr = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe_lr = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor_lr = ColumnTransformer([
    ("num", num_pipe_lr, num_features),
    ("cat", cat_pipe_lr, cat_features),
], remainder="drop")

In [ ]:
#Prétraitement des données qui vont servir pour le XGBoost

from sklearn.preprocessing import OrdinalEncoder

# Numérique : passthrough (aucun scaling)
# Catégoriel : entiers ; unknown_value=-1 protège des modalités absentes du train
cat_pipe_xgb = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor_xgb = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", cat_pipe_xgb, cat_features),
], remainder="drop")

In [ ]:
#Vérification
Xtr_lr  = preprocessor_lr.fit_transform(X_train)
Xte_lr  = preprocessor_lr.transform(X_test)

Xtr_xgb = preprocessor_xgb.fit_transform(X_train)
Xte_xgb = preprocessor_xgb.transform(X_test)

print("Logistique -> train:", Xtr_lr.shape,  "| test:", Xte_lr.shape)
print("XGBoost    -> train:", Xtr_xgb.shape, "| test:", Xte_xgb.shape)